In [ ]:
import os
import math
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import timm

class CFG:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    SAMPLE_SUB_CSV = os.path.join(ROOT_DIR, 'sample_submission.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'test_soundscapes')
    
    MODEL_PATH = '/kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/best_model_vit_fold_0.pth'
    KNN_PATH = '/kaggle/input/models/punyakdei/train-geo-alter2/pytorch/default/1/knn_spatial_fold_0.pkl'
    
    SR = 32000
    WINDOW_SECONDS = 5.0
    HOP_SECONDS = 2.5
    CHUNK_LENGTH = int(SR * WINDOW_SECONDS)
    HOP_LENGTH_AUDIO = int(SR * HOP_SECONDS)
    
    BACKBONE_NAME = 'vit_base_patch16_224'
    IMAGE_SIZE = (224, 224)
    
    CONFIDENCE_THRESHOLD = 0.85
    KNN_ALPHA = 0.3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:
class DualHeadViT(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE_NAME, num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, in_chans=3)
        in_features = self.backbone.head.in_features if hasattr(self.backbone, 'head') else 768
        self.backbone.reset_classifier(0)
        self.acoustic_head = nn.Sequential(nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, num_classes))
        self.geo_head = nn.Sequential(nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, 2))
    def forward(self, x):
        features = self.backbone(x)
        return self.acoustic_head(features), self.geo_head(features)

sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
NUM_CLASSES = len(submission_labels)

model = DualHeadViT(num_classes=NUM_CLASSES).to(device)
if os.path.exists(CFG.MODEL_PATH):
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
model.eval()

knn_model = None
if os.path.exists(CFG.KNN_PATH):
    knn_model = joblib.load(CFG.KNN_PATH)

mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=2048, hop_length=512, n_mels=128, f_min=20, f_max=16000).to(device)



In [ ]:
test_files = sorted(glob.glob(f"{CFG.SOUNDSCAPE_DIR}/*.ogg"))
if not test_files:
    test_files = sorted(glob.glob(os.path.join(CFG.ROOT_DIR, 'train_soundscapes', '*.ogg')))[:2]

all_predictions = []
all_row_ids = []

for audio_path in tqdm(test_files, desc="Inference"):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    data, _ = sf.read(audio_path, dtype='float32')
    if data.ndim > 1: data = data.mean(axis=1)
    
    total_samples = len(data)
    num_submission_chunks = math.ceil(total_samples / CFG.CHUNK_LENGTH)
    submission_block_probs = np.zeros((num_submission_chunks, NUM_CLASSES))
    
    # 1. Overlapping Windows Extraction (Safe End-Padding)
    windows, starts = [], []
    for start in range(0, total_samples, CFG.HOP_LENGTH_AUDIO):
        end = start + CFG.CHUNK_LENGTH
        if end > total_samples:
            slice_data = data[start:]
            pad_len = CFG.CHUNK_LENGTH - len(slice_data)
            padded = np.pad(slice_data, (0, pad_len))
            windows.append(padded)
            starts.append(start)
            break
        windows.append(data[start:end])
        starts.append(start)
        
    if not windows:
        windows.append(np.pad(data, (0, CFG.CHUNK_LENGTH - total_samples)))
        starts.append(0)
        
        # 2. Fully Streamed VRAM & RAM Safe Inference
    ac_out_list, geo_out_list = [], []
    sub_batch_size = 64
    
    # Pre-allocate mean and std on GPU
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
    
    # Enable all CPU cores for KNN if it exists
    if knn_model is not None:
        knn_model.n_jobs = -1
        
    for i in range(0, len(windows), sub_batch_size):
        sub_wins = windows[i:i+sub_batch_size]
        sub_waveforms = torch.tensor(np.array(sub_wins), dtype=torch.float32).to(device)
        
        with torch.no_grad():
            sub_mels = mel_transform(sub_waveforms)
            log_mels = torch.log(sub_mels + 1e-6)
            
            m_min = log_mels.reshape(len(sub_wins), -1).min(dim=1)[0].reshape(-1, 1, 1)
            m_max = log_mels.reshape(len(sub_wins), -1).max(dim=1)[0].reshape(-1, 1, 1)
            log_mels = (log_mels - m_min) / (m_max - m_min + 1e-6)
            
            sub_images = log_mels.unsqueeze(1)
            sub_images = F.interpolate(sub_images, size=CFG.IMAGE_SIZE, mode='bilinear', align_corners=False).repeat(1, 3, 1, 1)
            sub_images = (sub_images - mean) / std
            
            ac_out_sub, geo_out_sub = model(sub_images)
            
            ac_out_list.append(torch.sigmoid(ac_out_sub).cpu().numpy())
            geo_out_list.append(geo_out_sub.cpu().numpy())
            
    probs_visual = np.concatenate(ac_out_list, axis=0)
    predicted_coords = np.concatenate(geo_out_list, axis=0)
    
    # 3. Dynamic Self-Distillation Spatial Prior (Vectorized for Speed)
    if knn_model is not None:
        probs_knn = knn_model.predict(predicted_coords)
        max_probs = np.max(probs_visual, axis=1, keepdims=True)
        mask = (max_probs > CFG.CONFIDENCE_THRESHOLD)
        final_probs = np.where(mask, probs_visual, probs_visual * (1 - CFG.KNN_ALPHA) + probs_knn * CFG.KNN_ALPHA)
    else:
        final_probs = probs_visual

            
    # 4. Max-Pooling Aggregation per 5s submission block
    for i in range(num_submission_chunks):
        block_start = i * CFG.CHUNK_LENGTH
        block_end = block_start + CFG.CHUNK_LENGTH
        
        intersecting_probs = []
        for w_idx, w_start in enumerate(starts):
            w_end = w_start + CFG.CHUNK_LENGTH
            overlap = min(block_end, w_end) - max(block_start, w_start)
            if overlap > (CFG.CHUNK_LENGTH * 0.4): # >40% overlap
                intersecting_probs.append(final_probs[w_idx])
                
        if intersecting_probs:
            submission_block_probs[i] = np.max(intersecting_probs, axis=0)
            
        all_row_ids.append(f"{filename}_{(i+1)*5}")
        all_predictions.append(submission_block_probs[i])

# 5. Robust Merge against missing/extra rows for Kaggle Error Prevention
sub_df = pd.DataFrame(all_predictions, columns=submission_labels)
sub_df.insert(0, 'row_id', all_row_ids)

final_sub = pd.merge(sample_sub[['row_id']], sub_df, on='row_id', how='left').fillna(0.0)
final_sub.to_csv('submission.csv', index=False)
print("Submission formatted completely and merged safely!")

\n